# Course: Large Language Models - Assignment 2
## Task 1: Optimizer Performance on Non-Convex Functions

**Group Name:** Group 02  
**Optimizers Implemented from Scratch:**
1. Gradient Descent
2. Stochastic Gradient Descent with Momentum
3. Adam
4. RMSprop
5. AdaGrad

**Functions Evaluated:**
1. **Rosenbrock (2D):** $f(x, y) = (a - x)^2 + b(y - x^2)^2$ where $a=1, b=100$
2. **Ackley (1D Simplified):** $f(x) = -20\exp(-0.2\sqrt{x^2}) - \exp(\cos(2\pi x)) + 20 + e$ (with $x=0$ domain handling)


In [ ]:
# Cell 1: Environment Setup
import numpy as np
import time
import plotly.graph_objects as go

print("Environment set up successfully!")

In [ ]:
# Cell 2: Target Non-Convex Functions & Analytical Gradients
# --- Function 1: Rosenbrock (2D) ---
def f_rosenbrock(pos):
    x, y = pos[0], pos[1]
    return (1.0 - x)**2 + 100.0 * (y - x**2)**2

def grad_rosenbrock(pos):
    x, y = pos[0], pos[1]
    dx = -2.0 * (1.0 - x) - 400.0 * x * (y - x**2)
    dy = 200.0 * (y - x**2)
    return np.clip(np.array([dx, dy], dtype=np.float64), -1e4, 1e4)

# --- Function 2: Ackley (1D Simplified with x=0 safeguard) ---
def f_ackley(pos):
    x = pos[0]
    # Handle x = 0 by setting f(0) = 0 as per assignment prompt
    if np.isclose(x, 0.0, atol=1e-12):
        return 0.0
    safe_x = np.sqrt(x**2 + 1e-15)
    return -20.0 * np.exp(-0.2 * safe_x) - np.exp(np.cos(2.0 * np.pi * x)) + 20.0 + np.e

def grad_ackley(pos):
    x = pos[0]
    if np.isclose(x, 0.0, atol=1e-12):
        return np.array([0.0], dtype=np.float64)
    
    safe_abs_x = np.sqrt(x**2 + 1e-15)
    dx_abs = x / safe_abs_x
    term1_grad = 4.0 * dx_abs * np.exp(-0.2 * safe_abs_x)
    term2_grad = 2.0 * np.pi * np.sin(2.0 * np.pi * x) * np.exp(np.cos(2.0 * np.pi * x))
    return np.clip(np.array([term1_grad + term2_grad], dtype=np.float64), -1e4, 1e4)

In [ ]:
# Cell 3: Scratch Optimizers Implementation
# 1. Gradient Descent
def sgd(grad_fn, start_pt, lr, max_iters=1000, tol=1e-5):
    pos = np.array(start_pt, dtype=np.float64)
    history = [pos.copy()]
    for _ in range(max_iters):
        g = grad_fn(pos)
        if np.isnan(g).any() or np.linalg.norm(g) < tol:
            break
        pos -= lr * g
        pos = np.clip(pos, -1e3, 1e3)
        history.append(pos.copy())
    return np.array(history)

# 2. Stochastic Gradient Descent with Momentum
def momentum_gd(grad_fn, start_pt, lr, beta=0.9, max_iters=1000, tol=1e-5):
    pos = np.array(start_pt, dtype=np.float64)
    v = np.zeros_like(pos, dtype=np.float64)
    history = [pos.copy()]
    for _ in range(max_iters):
        g = grad_fn(pos)
        if np.isnan(g).any() or np.linalg.norm(g) < tol:
            break
        v = beta * v + lr * g
        pos -= v
        pos = np.clip(pos, -1e3, 1e3)
        history.append(pos.copy())
    return np.array(history)

# 3. Adam
def adam(grad_fn, start_pt, lr, beta1=0.9, beta2=0.999, eps=1e-8, max_iters=1000, tol=1e-5):
    pos = np.array(start_pt, dtype=np.float64)
    m = np.zeros_like(pos, dtype=np.float64)
    v = np.zeros_like(pos, dtype=np.float64)
    history = [pos.copy()]
    for t in range(1, max_iters + 1):
        g = grad_fn(pos)
        if np.isnan(g).any() or np.linalg.norm(g) < tol:
            break
        m = beta1 * m + (1.0 - beta1) * g
        v = beta2 * v + (1.0 - beta2) * (g**2)
        m_hat = m / (1.0 - beta1**t)
        v_hat = v / (1.0 - beta2**t)
        pos -= (lr / (np.sqrt(v_hat) + eps)) * m_hat
        history.append(pos.copy())
    return np.array(history)

# 4. RMSprop
def rmsprop(grad_fn, start_pt, lr, beta=0.9, eps=1e-8, max_iters=1000, tol=1e-5):
    pos = np.array(start_pt, dtype=np.float64)
    v = np.zeros_like(pos, dtype=np.float64)
    history = [pos.copy()]
    for _ in range(max_iters):
        g = grad_fn(pos)
        if np.isnan(g).any() or np.linalg.norm(g) < tol:
            break
        v = beta * v + (1.0 - beta) * (g**2)
        pos -= (lr / (np.sqrt(v) + eps)) * g
        history.append(pos.copy())
    return np.array(history)

# 5. AdaGrad
def adagrad(grad_fn, start_pt, lr, eps=1e-8, max_iters=1000, tol=1e-5):
    pos = np.array(start_pt, dtype=np.float64)
    G = np.zeros_like(pos, dtype=np.float64)
    history = [pos.copy()]
    for _ in range(max_iters):
        g = grad_fn(pos)
        if np.isnan(g).any() or np.linalg.norm(g) < tol:
            break
        G += g**2
        pos -= (lr / (np.sqrt(G) + eps)) * g
        history.append(pos.copy())
    return np.array(history)

In [ ]:
# Cell 4: Benchmark Execution & Hoverable Convergence Graphs
optimizers = {
    'Gradient Descent': sgd,
    'SGD with Momentum': momentum_gd,
    'Adam': adam,
    'RMSprop': rmsprop,
    'AdaGrad': adagrad
}

learning_rates = [0.01, 0.05, 0.1]
max_iters = 1000
tol = 1e-5

tasks = [
    ("Rosenbrock (2D)", f_rosenbrock, grad_rosenbrock, [-1.5, 1.5]),
    ("Ackley (1D)", f_ackley, grad_ackley, [2.5])
]

for fn_name, fn, grad_fn, start_point in tasks:
    print("=" * 86)
    print(f"  BENCHMARK RESULTS FOR TASK 1: {fn_name.upper()}")
    print("=" * 86)
    
    for lr in learning_rates:
        print(f"\n>>> Fixed Learning Rate (alpha): {lr} | Termination: ||grad|| < {tol} OR Max Iters = {max_iters}")
        if len(start_point) == 2:
            print(f"{'Optimizer':<22} | {'Optimal x*':<10} | {'Optimal y*':<10} | {'f(x*, y*)':<13} | {'Time (s)':<10} | {'Iters':<6}")
            print("-" * 83)
        else:
            print(f"{'Optimizer':<22} | {'Optimal x*':<12} | {'f(x*)':<13} | {'Time (s)':<10} | {'Iters':<6}")
            print("-" * 71)
        
        fig = go.Figure()
        
        for name, opt_func in optimizers.items():
            t0 = time.perf_counter()
            hist = opt_func(grad_fn, start_point, lr, max_iters=max_iters, tol=tol)
            t1 = time.perf_counter()
            
            elapsed_time = t1 - t0
            final_pos = hist[-1]
            
            try:
                final_val = fn(final_pos)
            except OverflowError:
                final_val = np.nan
                
            num_iters = len(hist) - 1
            
            if len(start_point) == 2:
                print(f"{name:<22} | {final_pos[0]:<10.4f} | {final_pos[1]:<10.4f} | {final_val:<13.6e} | {elapsed_time:<10.6f} | {num_iters:<6}")
            else:
                print(f"{name:<22} | {final_pos[0]:<12.4f} | {final_val:<13.6e} | {elapsed_time:<10.6f} | {num_iters:<6}")
            
            loss_history = []
            for pt in hist:
                try:
                    val = fn(pt)
                    loss_history.append(val if np.isfinite(val) and val < 1e12 else np.nan)
                except OverflowError:
                    loss_history.append(np.nan)
                    
            fig.add_trace(go.Scatter(
                x=list(range(len(loss_history))),
                y=loss_history,
                mode='lines',
                name=name,
                hovertemplate="<b>" + name + "</b><br>" +
                              "Iteration: %{x}<br>" +
                              "Loss: %{y:.6e}<extra></extra>"
            ))

        fig.update_layout(
            title=f"Interactive Convergence: {fn_name} (α={lr})",
            xaxis_title="Iteration Step",
            yaxis_title="Loss f(x) [Log Scale]",
            yaxis_type="log",
            hovermode="x unified",
            template="plotly_white"
        )
        fig.show()